# MCQA Circuit Tracing with Gemma-2-2B CLTs

This notebook mirrors the upstream `circuit-tracer` demos, but uses representative MCQA prompts and the Gemma-2-2B CLT weights by default.

It covers the three intended library tasks: attribution graph creation, graph visualization, and feature intervention.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(repo_root / "src"))

from circuit_tracing_ot.config import resolve_transcoder_set
from circuit_tracing_ot.interventions import FeatureIntervention, run_feature_intervention
from circuit_tracing_ot.mcqa_prompts import load_prompts
from circuit_tracing_ot.model import load_replacement_model
from circuit_tracing_ot.trace import TraceConfig, trace_prompt
from circuit_tracing_ot.server import serve_graph_files

## 1. Load Gemma-2-2B with CLTs

Use `426k` for the smaller CLT, or switch to `2.5m` for the larger pretrained CLT.

In [ ]:
prompts = load_prompts(dataset_split="train", limit=1)
prompt = prompts[0]
transcoder_set = resolve_transcoder_set(transcoder_set=None, transcoder_size="426k")

model = load_replacement_model(
    model_name="google/gemma-2-2b",
    transcoder_set=transcoder_set,
    dtype_name="bf16",
)

print(prompt.prompt)
print("expected:", repr(prompt.expected_answer))
print("transcoder_set:", transcoder_set)

## 2. Build and Export an Attribution Graph

This wraps the same calls used in `attribute_demo.ipynb`: `attribute`, `graph.to_pt`, and `create_graph_files`.

In [ ]:
config = TraceConfig(
    model_name="google/gemma-2-2b",
    transcoder_set=transcoder_set,
    dtype="bf16",
    batch_size=128,
    max_n_logits=10,
    desired_logit_prob=0.95,
    node_threshold=0.8,
    edge_threshold=0.98,
)

result = trace_prompt(model=model, prompt=prompt, config=config)
result

## 3. Serve the Interactive HTML Graph

Open the displayed local URL. In the UI, click nodes to inspect them, pin nodes with Cmd/Ctrl-click, edit annotations, and group nodes into supernodes.

In [ ]:
port = 8046
server = serve_graph_files(graph_file_dir="graph_files", port=port)
print(f"Open http://localhost:{port}/index.html")

Stop the server when finished.

In [ ]:
# server.stop()

## 4. Feature Interventions

After selecting features from the rendered graph, replace the placeholder below with `(layer, position, feature_idx, value)` tuples. Position `-1` means the final prompt position, matching the upstream intervention demo.

In [ ]:
# Replace this with a feature selected from the graph UI.
interventions = [
    # FeatureIntervention(layer=20, position=-1, feature_idx=341, value=0.0),
]

if interventions:
    intervention_result = run_feature_intervention(
        model=model,
        prompt=prompt.prompt,
        interventions=interventions,
        top_k=5,
    )
    intervention_result
else:
    print("Choose one or more feature IDs from the graph, then rerun this cell.")